Khushi Khatri nlp exp 9 beb222 

Aim: Implement Text Similarity Recognizer for the chosen text documents.

Theory:
Text similarity measures how alike two pieces of text are. Unlike the
earlier experiments in this project (which mostly analyzed one document
at a time), similarity is always a comparison between a pair of
documents, and different measures capture different notions of
"alike".

Text similarity is what powers: finding duplicate or near-duplicate
reviews (spam/fake-review detection), recommending listings with
similar guest feedback, and clustering reviews that talk about the same
things. This experiment implements and compares all three measures, then
uses cosine similarity to build a simple "find similar reviews" search
over the dataset.


In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity

STOPWORDS = set(stopwords.words('english'))
pd.set_option('display.max_colwidth', 120)

[nltk_data] Error loading punkt: <urlopen error [Errno 104] Connection
[nltk_data]     reset by peer>
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv('Balanced_Airbnb_Reviews_Dataset.csv')
print('Shape:', df.shape)
df[['review_id', 'review_text', 'sentiment_label']].head()

Shape: (15000, 42)


,review_id,review_text,sentiment_label
0,369314882,Amazing stay! The place felt very cozy for 4 guests. Check-in was smooth and the amenities were exactly what we needed.,positive
1,490116563,It was okay for the price. Location in XIII Aurelia was convenient enough.,neutral
2,582235668,Loved every minute of it. Our superhost was super responsive and easy to communicate with. Great location in Venusti...,positive
3,68054683,"Decent stay overall. Some things could be improved, like cleanliness in the entire apartment.",neutral
4,248483824,Reasonable for a short trip. Location in Long Island City was convenient enough.,neutral


In [3]:
def get_token_set(text):
    tokens = [w.lower() for w in word_tokenize(text) if w.isalpha() and w.lower() not in STOPWORDS]
    return set(tokens)


def jaccard_similarity(text_a, text_b):
    set_a, set_b = get_token_set(text_a), get_token_set(text_b)
    if not set_a and not set_b:
        return 0.0
    intersection = set_a & set_b
    union = set_a | set_b
    return len(intersection) / len(union)


# demo
doc_a = df['review_text'].iloc[0]
doc_b = df['review_text'].iloc[1]
print('DOC A:', doc_a)
print('DOC B:', doc_b)
print('Jaccard similarity:', round(jaccard_similarity(doc_a, doc_b), 4))

DOC A: Amazing stay! The place felt very cozy for 4 guests. Check-in was smooth and the amenities were exactly what we needed.
DOC B: It was okay for the price. Location in XIII Aurelia was convenient enough.
Jaccard similarity: 0.0


In [4]:
def cosine_similarity_pair(text_a, text_b):
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform([text_a, text_b])
    sim_matrix = sk_cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
    return sim_matrix[0][0]


print('Cosine similarity:', round(cosine_similarity_pair(doc_a, doc_b), 4))


Cosine similarity: 0.0


In [5]:
def levenshtein_distance(s1, s2):
    """Minimum number of single-character edits (insert/delete/substitute)
    to turn s1 into s2, computed via classic dynamic programming."""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],      # deletion
                    dp[i][j - 1],      # insertion
                    dp[i - 1][j - 1],  # substitution
                )
    return dp[m][n]


def levenshtein_similarity(s1, s2):
    """Normalize edit distance into a 0-1 similarity score (1 = identical)."""
    dist = levenshtein_distance(s1, s2)
    max_len = max(len(s1), len(s2))
    if max_len == 0:
        return 1.0
    return 1 - (dist / max_len)


print('Edit distance:', levenshtein_distance(doc_a, doc_b))
print('Edit-distance similarity:', round(levenshtein_similarity(doc_a, doc_b), 4))

Edit distance: 89
Edit-distance similarity: 0.2521


In [6]:
pairs = [
    ('Identical text', doc_a, doc_a),
    ('Same review, reworded slightly', doc_a, doc_a.replace('Amazing', 'Wonderful')),
    ('Two random different reviews', doc_a, doc_b),
    ('Two reviews with opposite sentiment',
     df[df['sentiment_label'] == 'positive']['review_text'].iloc[0],
     df[df['sentiment_label'] == 'negative']['review_text'].iloc[0]),
]

results = []
for label, a, b in pairs:
    results.append({
        'pair': label,
        'jaccard': round(jaccard_similarity(a, b), 4),
        'cosine': round(cosine_similarity_pair(a, b), 4),
        'levenshtein_similarity': round(levenshtein_similarity(a, b), 4),
    })

pd.DataFrame(results)

,pair,jaccard,cosine,levenshtein_similarity
0,Identical text,1.0000,1.0000,1.0000
1,"Same review, reworded slightly",0.8182,0.8350,0.9256
2,Two random different reviews,0.0000,0.0000,0.2521
3,Two reviews with opposite sentiment,0.0435,0.0826,0.2356


In [7]:
SAMPLE_SIZE = 2000  # keep this manageable; full dataset works the same way, just slower to vectorize/compare
sample_df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

search_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
search_matrix = search_vectorizer.fit_transform(sample_df['review_text'])

print('TF-IDF matrix shape:', search_matrix.shape)

TF-IDF matrix shape: (2000, 351)


In [8]:
def find_similar_reviews(query_text, top_n=5):
    """Vectorize the query with the same fitted TF-IDF vectorizer, then
    rank every review in sample_df by cosine similarity to it."""
    query_vec = search_vectorizer.transform([query_text])
    similarities = sk_cosine_similarity(query_vec, search_matrix)[0]

    top_indices = similarities.argsort()[::-1][:top_n]
    return pd.DataFrame({
        'review_text': sample_df.loc[top_indices, 'review_text'].values,
        'sentiment_label': sample_df.loc[top_indices, 'sentiment_label'].values,
        'similarity': similarities[top_indices].round(4),
    })


query = "The apartment was spotless and the host was very responsive, would definitely stay again."
print('QUERY:', query, '\n')
find_similar_reviews(query, top_n=5)

QUERY: The apartment was spotless and the host was very responsive, would definitely stay again. 



,review_text,sentiment_label,similarity
0,Exceeded our expectations! The entire apartment was spotless and exactly as described. We would definitely book this...,positive,0.5393
1,Loved every minute of it. We would definitely book this entire apartment again. The host was super responsive and ea...,positive,0.5276
2,Amazing stay! The entire apartment was spotless and exactly as described. The host was super responsive and easy to ...,positive,0.5055
3,Amazing stay! The entire apartment was spotless and exactly as described. We would definitely book this entire apart...,positive,0.5048
4,Amazing stay! We would definitely book this entire apartment again. The entire apartment was spotless and exactly as...,positive,0.4643


In [9]:
full_similarity_matrix = sk_cosine_similarity(search_matrix)
np.fill_diagonal(full_similarity_matrix, -1)  # exclude self-matches

best_match_idx = full_similarity_matrix.argmax(axis=1)
best_match_score = full_similarity_matrix.max(axis=1)

sample_df['most_similar_review'] = sample_df.loc[best_match_idx, 'review_text'].values
sample_df['similarity_score'] = best_match_score.round(4)

sample_df[['review_text', 'most_similar_review', 'similarity_score']].sort_values(
    'similarity_score', ascending=False
).head(10)

,review_text,most_similar_review,similarity_score
1999,"Decent stay overall. The host was polite but slow to respond a couple of times. Some things could be improved, like ...","Decent stay overall. The host was polite but slow to respond a couple of times. Some things could be improved, like ...",1.0
1242,Reasonable for a short trip. The host was polite but slow to respond a couple of times.,Reasonable for a short trip. The host was polite but slow to respond a couple of times.,1.0
1275,It was okay for the price. It served its purpose for our stay in Sydney.,It was okay for the price. It served its purpose for our stay in Sydney.,1.0
1269,Wonderful experience. The place felt very cozy for 2 guests. Our superhost was super responsive and easy to communic...,Wonderful experience. The place felt very cozy for 4 guests. Our superhost was super responsive and easy to communic...,1.0
1266,"Nothing special, but fine. Location in Enclos-St-Laurent was convenient enough.","Nothing special, but fine. Location in Enclos-St-Laurent was convenient enough.",1.0
523,Decent stay overall. The private room in apartment was fine but a little smaller than expected.,Decent stay overall. The private room in apartment was fine but a little smaller than expected.,1.0
1261,Amazing stay! The place felt very cozy for 6 guests. Check-in was smooth and the amenities were exactly what we need...,Amazing stay! Check-in was smooth and the amenities were exactly what we needed. The place felt very cozy for 4 gues...,1.0
1256,Reasonable for a short trip. It served its purpose for our stay in Bangkok.,Reasonable for a short trip. It served its purpose for our stay in Bangkok.,1.0
1254,We had a rough experience. The private room in apartment was not as clean as the photos suggested. Check-in was conf...,We had a rough experience. The private room in apartment was not as clean as the photos suggested. Our superhost was...,1.0
1247,Amazing stay! We would definitely book this private room in apartment again. The private room in apartment was spotl...,Amazing stay! The private room in apartment was spotless and exactly as described. We would definitely book this pri...,1.0


In [10]:
sample_df.to_csv('Similarity_Airbnb_Reviews.csv', index=False)
print('Saved to Similarity_Airbnb_Reviews.csv')
print(sample_df.shape)

Saved to Similarity_Airbnb_Reviews.csv
(2000, 44)
